# Football Dataset - Exploratory Data Analysis

This notebook explores the original read-only football dataset for object detection.

**Dataset Location:** `/cluster/projects/vc/courses/TDT17/other/Football2025/`

**Classes:**
- 0: player
- 1: ball

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import cv2
from collections import defaultdict, Counter
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Dataset paths
DATASET_ROOT = Path('/cluster/projects/vc/courses/TDT17/other/Football2025')

## 1. Dataset Structure Overview

In [ ]:
# List all matches
matches = [d for d in DATASET_ROOT.iterdir() if d.is_dir()]
print(f"Total matches: {len(matches)}")
print("\nMatch directories:")
for match in sorted(matches):
    print(f"  - {match.name}")

In [ ]:
# Quick summary of dataset contents
def summarize_match(match_path):
    """Summarize contents of a match directory."""
    match_path = Path(match_path)
    summary = {
        'name': match_path.name,
        'has_video': False,
        'has_annotations': False,
        'has_images': False,
        'has_labels': False,
        'num_images': 0,
        'num_labels': 0,
        'video_files': [],
        'structure_type': 'unknown'
    }
    
    # Check for common files
    for item in match_path.rglob('*'):
        if item.is_file():
            if item.suffix.lower() in ['.mp4', '.avi', '.mov']:
                summary['has_video'] = True
                summary['video_files'].append(item.name)
            elif item.name == 'annotations.xml':
                summary['has_annotations'] = True
            elif item.suffix.lower() in ['.png', '.jpg', '.jpeg'] and 'images' in str(item):
                summary['has_images'] = True
                summary['num_images'] += 1
            elif item.suffix == '.txt' and 'labels' in str(item):
                summary['has_labels'] = True
                summary['num_labels'] += 1
    
    # Determine structure type
    if 'BODO' in match_path.name:
        summary['structure_type'] = 'multi-part'
    else:
        summary['structure_type'] = 'single'
    
    return summary

# Summarize all matches
print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

summaries = []
for match in sorted(matches):
    summary = summarize_match(match)
    summaries.append(summary)
    
    print(f"\n📂 {summary['name']}")
    print(f"   Type: {summary['structure_type']}")
    print(f"   Images: {summary['num_images']:,}")
    print(f"   Labels: {summary['num_labels']:,}")
    print(f"   Annotations XML: {'✓' if summary['has_annotations'] else '✗'}")
    print(f"   Videos: {len(summary['video_files'])} files")
    if summary['video_files']:
        for vid in summary['video_files'][:3]:  # Show first 3
            print(f"      - {vid}")
        if len(summary['video_files']) > 3:
            print(f"      ... and {len(summary['video_files']) - 3} more")

# Total summary
total_images = sum(s['num_images'] for s in summaries)
total_labels = sum(s['num_labels'] for s in summaries)
total_videos = sum(len(s['video_files']) for s in summaries)

print("\n" + "=" * 80)
print("TOTALS")
print("=" * 80)
print(f"Total Images: {total_images:,}")
print(f"Total Labels: {total_labels:,}")
print(f"Total Videos: {total_videos}")
print(f"Matches with annotations.xml: {sum(s['has_annotations'] for s in summaries)}")
print("=" * 80)

In [ ]:
# Comprehensive directory structure exploration
def explore_directory(path, max_depth=3, current_depth=0, prefix=""):
    """
    Explore directory structure showing folders and file counts.
    """
    if current_depth >= max_depth:
        return
    
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return
    
    try:
        items = sorted(path.iterdir())
        dirs = [item for item in items if item.is_dir()]
        files = [item for item in items if item.is_file()]
        
        # Print current directory info
        if current_depth > 0:
            print(f"{prefix}├── 📁 {path.name}/")
            print(f"{prefix}│   Files: {len(files)}, Subdirs: {len(dirs)}")
            
            # Show file types and counts
            if files:
                file_types = defaultdict(int)
                total_size = 0
                for f in files:
                    ext = f.suffix.lower() or 'no_extension'
                    file_types[ext] += 1
                    try:
                        total_size += f.stat().st_size
                    except:
                        pass
                
                print(f"{prefix}│   File types: ", end="")
                type_strs = [f"{ext}({count})" for ext, count in sorted(file_types.items())]
                print(", ".join(type_strs))
                
                if total_size > 0:
                    size_mb = total_size / (1024**2)
                    size_gb = total_size / (1024**3)
                    if size_gb > 1:
                        print(f"{prefix}│   Total size: {size_gb:.2f} GB")
                    else:
                        print(f"{prefix}│   Total size: {size_mb:.1f} MB")
        
        # Recurse into subdirectories
        for i, subdir in enumerate(dirs):
            is_last = (i == len(dirs) - 1)
            new_prefix = prefix + ("│   " if current_depth > 0 else "")
            explore_directory(subdir, max_depth, current_depth + 1, new_prefix)
            
    except PermissionError:
        print(f"{prefix}│   [Permission Denied]")

print("=" * 80)
print("COMPLETE DATASET STRUCTURE")
print("=" * 80)
print("\nExploring all match directories...\n")

for match in sorted(matches):
    print("\n" + "=" * 80)
    print(f"📂 {match.name}")
    print("=" * 80)
    explore_directory(match, max_depth=4)
    print()

print("=" * 80)

In [ ]:
# Check image dimensions across the dataset
print("Analyzing image dimensions...")
print("This will sample images from each match to check sizes.\n")

image_dimensions = []

for match in sorted(matches):
    print(f"Checking {match.name}...")
    
    # Find all image files in this match
    image_files = list(match.rglob('*.png')) + list(match.rglob('*.jpg'))
    
    if not image_files:
        print(f"  No images found")
        continue
    
    # Sample images to check (check first 10, middle 10, and last 10)
    sample_size = min(30, len(image_files))
    if len(image_files) > 30:
        # Sample from beginning, middle, and end
        samples = (
            image_files[:10] +
            image_files[len(image_files)//2 - 5:len(image_files)//2 + 5] +
            image_files[-10:]
        )
    else:
        samples = image_files[:sample_size]
    
    match_dims = []
    for img_file in samples:
        try:
            # Use PIL to get dimensions without loading full image
            with Image.open(img_file) as img:
                width, height = img.size
                match_dims.append({
                    'match': match.name,
                    'width': width,
                    'height': height,
                    'resolution': f"{width}x{height}",
                    'aspect_ratio': width / height if height > 0 else 0
                })
        except Exception as e:
            continue
    
    if match_dims:
        # Get unique dimensions for this match
        unique_dims = set((d['width'], d['height']) for d in match_dims)
        
        print(f"  Total images: {len(image_files):,}")
        print(f"  Sampled: {len(match_dims)}")
        print(f"  Dimensions found: {', '.join(f'{w}x{h}' for w, h in sorted(unique_dims))}")
        
        if len(unique_dims) > 1:
            print(f"  ⚠️  Multiple dimensions detected!")
        
        image_dimensions.extend(match_dims)
    
    print()

# Create DataFrame
df_dims = pd.DataFrame(image_dimensions)

print("=" * 80)
print("IMAGE DIMENSION SUMMARY")
print("=" * 80)

if len(df_dims) > 0:
    # Overall statistics
    print("\nAll Images Sampled:")
    print(f"  Total sampled: {len(df_dims):,}")
    print(f"  Unique dimensions: {df_dims['resolution'].nunique()}")
    
    print("\nDimension Distribution:")
    dim_counts = df_dims['resolution'].value_counts()
    for resolution, count in dim_counts.items():
        pct = count / len(df_dims) * 100
        print(f"  {resolution}: {count} ({pct:.1f}%)")
    
    print("\nBy Match:")
    for match_name in sorted(df_dims['match'].unique()):
        match_data = df_dims[df_dims['match'] == match_name]
        unique_dims = match_data['resolution'].unique()
        print(f"  {match_name}: {', '.join(unique_dims)}")
    
    print("\nStatistics:")
    print(f"  Mean width:  {df_dims['width'].mean():.0f} px")
    print(f"  Mean height: {df_dims['height'].mean():.0f} px")
    print(f"  Min width:   {df_dims['width'].min()} px")
    print(f"  Max width:   {df_dims['width'].max()} px")
    print(f"  Min height:  {df_dims['height'].min()} px")
    print(f"  Max height:  {df_dims['height'].max()} px")
    print(f"  Aspect ratio: {df_dims['aspect_ratio'].mean():.3f} (avg)")
    
    # Visualize
    if df_dims['resolution'].nunique() <= 10:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Dimension distribution
        dim_counts.plot(kind='bar', ax=axes[0])
        axes[0].set_xlabel('Resolution')
        axes[0].set_ylabel('Count (sampled)')
        axes[0].set_title('Image Resolution Distribution')
        axes[0].tick_params(axis='x', rotation=45)
        
        # By match
        match_dims = df_dims.groupby(['match', 'resolution']).size().unstack(fill_value=0)
        match_dims.plot(kind='bar', stacked=True, ax=axes[1])
        axes[1].set_xlabel('Match')
        axes[1].set_ylabel('Count (sampled)')
        axes[1].set_title('Image Dimensions by Match')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].legend(title='Resolution', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        plt.show()
    
else:
    print("No image data collected")

print("=" * 80)

## 1.5. Image Dimensions Analysis

Check the actual dimensions of all images in the dataset.

In [ ]:
"""
## CRITICAL: Frame-Label Numbering Verification

Check if PNG frames start at 1 while label files start at 0 (off-by-one error).
This could cause frames to be matched with the wrong labels.
"""

import re

def extract_frame_number(filename):
    """Extract frame number from filename like 'RBK-AALESUND_frame_000001.png'"""
    match = re.search(r'frame_(\d+)', filename)
    if match:
        return int(match.group(1))
    return None

def check_frame_label_alignment(match_path):
    """
    Check if frames and labels are properly aligned.
    Returns detailed information about the numbering.
    """
    match_path = Path(match_path)
    results = {
        'match': match_path.name,
        'issues_found': [],
        'frame_range': None,
        'label_range': None,
        'alignment_check': None
    }
    
    # Find image and label directories
    image_dirs = list(match_path.rglob('images/train'))
    label_dirs = list(match_path.rglob('labels/train'))
    
    if not image_dirs or not label_dirs:
        results['issues_found'].append("Could not find images/train or labels/train")
        return results
    
    image_dir = image_dirs[0]
    label_dir = label_dirs[0]
    
    # Get all frames and labels
    frames = sorted(image_dir.glob('*.png'))
    labels = sorted(label_dir.glob('*.txt'))
    
    if not frames or not labels:
        results['issues_found'].append("No frames or labels found")
        return results
    
    # Extract frame numbers
    frame_numbers = []
    frame_files = {}
    for f in frames:
        num = extract_frame_number(f.name)
        if num is not None:
            frame_numbers.append(num)
            frame_files[num] = f.name
    
    # Extract label numbers (assuming same naming pattern)
    label_numbers = []
    label_files = {}
    for l in labels:
        num = extract_frame_number(l.name)
        if num is not None:
            label_numbers.append(num)
            label_files[num] = l.name
    
    if not frame_numbers or not label_numbers:
        results['issues_found'].append("Could not extract frame numbers")
        return results
    
    # Check ranges
    results['frame_range'] = (min(frame_numbers), max(frame_numbers))
    results['label_range'] = (min(label_numbers), max(label_numbers))
    
    # Check if frame numbering starts at different point than labels
    if results['frame_range'][0] != results['label_range'][0]:
        offset = results['frame_range'][0] - results['label_range'][0]
        results['issues_found'].append(
            f"OFFSET DETECTED: Frames start at {results['frame_range'][0]}, "
            f"Labels start at {results['label_range'][0]} (offset: {offset})"
        )
    
    # Check specific examples
    examples = []
    check_nums = sorted(frame_numbers)[:5]  # Check first 5 frames
    
    for num in check_nums:
        frame_exists = num in frame_files
        label_exists = num in label_files
        label_minus_1_exists = (num - 1) in label_files
        
        example = {
            'frame_num': num,
            'frame_file': frame_files.get(num, 'MISSING'),
            'label_file': label_files.get(num, 'MISSING'),
            'label_minus_1': label_files.get(num - 1, 'MISSING'),
            'match': frame_exists and label_exists,
            'off_by_one': frame_exists and label_minus_1_exists and not label_exists
        }
        examples.append(example)
    
    results['examples'] = examples
    
    # Overall alignment check
    if all(e['match'] for e in examples):
        results['alignment_check'] = 'CORRECT: Frames and labels align perfectly'
    elif all(e['off_by_one'] for e in examples):
        results['alignment_check'] = 'OFF-BY-ONE ERROR: Frames need labels with number-1'
    else:
        results['alignment_check'] = 'MIXED: Some align, some don\'t'
    
    return results

print("=" * 80)
print("FRAME-LABEL ALIGNMENT CHECK")
print("=" * 80)
print("\nChecking if PNG frame numbering matches TXT label file numbering...")
print("This will detect off-by-one errors where frames start at 1 but labels start at 0.\n")

all_results = []

for match in sorted(matches):
    print(f"\n{'=' * 80}")
    print(f"📂 {match.name}")
    print('=' * 80)
    
    results = check_frame_label_alignment(match)
    all_results.append(results)
    
    # Print findings
    print(f"Frame range: {results['frame_range']}")
    print(f"Label range: {results['label_range']}")
    
    if results['issues_found']:
        print(f"\n⚠️  ISSUES FOUND:")
        for issue in results['issues_found']:
            print(f"  - {issue}")
    
    print(f"\nAlignment: {results['alignment_check']}")
    
    if 'examples' in results:
        print(f"\nFirst 5 frames check:")
        print(f"{'Frame#':<10} {'Frame File':<35} {'Label File':<35} {'Status':<20}")
        print('-' * 100)
        for ex in results['examples']:
            status = ''
            if ex['match']:
                status = '✓ MATCH'
            elif ex['off_by_one']:
                status = '⚠️  OFF-BY-ONE'
            else:
                status = '✗ MISMATCH'
            
            frame_file = ex['frame_file'] if ex['frame_file'] != 'MISSING' else '❌ MISSING'
            label_file = ex['label_file'] if ex['label_file'] != 'MISSING' else '❌ MISSING'
            
            print(f"{ex['frame_num']:<10} {frame_file:<35} {label_file:<35} {status:<20}")
            
            # If off-by-one, show what label exists
            if ex['off_by_one']:
                print(f"{'':<10} {'':<35} {'→ ' + ex['label_minus_1']:<35} {'(label n-1 exists)':<20}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

# Count issues
matches_with_offset = sum(1 for r in all_results if any('OFFSET DETECTED' in i for i in r.get('issues_found', [])))
matches_with_offbyone = sum(1 for r in all_results if r.get('alignment_check', '').startswith('OFF-BY-ONE'))
matches_correct = sum(1 for r in all_results if r.get('alignment_check', '').startswith('CORRECT'))

print(f"\nTotal matches checked: {len(all_results)}")
print(f"Matches with correct alignment: {matches_correct}")
print(f"Matches with off-by-one error: {matches_with_offbyone}")
print(f"Matches with frame/label range offset: {matches_with_offset}")

if matches_with_offbyone > 0:
    print("\n" + "!" * 80)
    print("⚠️  CRITICAL ISSUE DETECTED!")
    print("!" * 80)
    print("\nOff-by-one errors found! This means:")
    print("  - Frame N is being paired with Label N")
    print("  - But Label N should be for Frame N+1")
    print("  - This causes ALL frames to have WRONG labels!")
    print("\nAffected matches:")
    for r in all_results:
        if r.get('alignment_check', '').startswith('OFF-BY-ONE'):
            print(f"  - {r['match']}")
    print("\n⚠️  The local dataset (football_dataset_complete) needs to be regenerated!")
    print("!" * 80)

print("=" * 80)

In [ ]:
"""
Check if number of images matches number of labels for each match
This will tell us if there are missing labels or images.
"""

print("=" * 80)
print("IMAGE vs LABEL COUNT VERIFICATION")
print("=" * 80)
print("\nChecking if each match has the same number of images and labels...\n")

mismatches = []

for match in sorted(matches):
    print(f"📂 {match.name}")
    
    # Find image and label directories
    image_dirs = list(match.rglob('images/train'))
    label_dirs = list(match.rglob('labels/train'))
    
    if not image_dirs:
        print(f"  ⚠️  No images/train directory found")
        continue
    
    if not label_dirs:
        print(f"  ⚠️  No labels/train directory found")
        continue
    
    # Count files
    image_files = list(image_dirs[0].glob('*.png')) + list(image_dirs[0].glob('*.jpg'))
    label_files = list(label_dirs[0].glob('*.txt'))
    
    num_images = len(image_files)
    num_labels = len(label_files)
    
    print(f"  Images: {num_images:,}")
    print(f"  Labels: {num_labels:,}")
    
    if num_images == num_labels:
        print(f"  ✓ MATCH - Counts are equal")
    else:
        diff = num_images - num_labels
        print(f"  ✗ MISMATCH - Difference: {diff:+d}")
        if diff > 0:
            print(f"     → {abs(diff)} more images than labels")
        else:
            print(f"     → {abs(diff)} more labels than images")
        
        mismatches.append({
            'match': match.name,
            'images': num_images,
            'labels': num_labels,
            'difference': diff
        })
    
    # Check frame number ranges
    if image_files and label_files:
        image_nums = []
        for img in image_files:
            num = extract_frame_number(img.name)
            if num is not None:
                image_nums.append(num)
        
        label_nums = []
        for lbl in label_files:
            num = extract_frame_number(lbl.name)
            if num is not None:
                label_nums.append(num)
        
        if image_nums and label_nums:
            img_range = (min(image_nums), max(image_nums))
            lbl_range = (min(label_nums), max(label_nums))
            
            print(f"  Image frame range: {img_range[0]:,} to {img_range[1]:,}")
            print(f"  Label frame range: {lbl_range[0]:,} to {lbl_range[1]:,}")
            
            if img_range != lbl_range:
                print(f"  ⚠️  WARNING: Frame ranges don't match!")
                print(f"     Image range span: {img_range[1] - img_range[0] + 1}")
                print(f"     Label range span: {lbl_range[1] - lbl_range[0] + 1}")
    
    print()

print("=" * 80)
print("SUMMARY")
print("=" * 80)

if mismatches:
    print(f"\n⚠️  FOUND {len(mismatches)} MATCHES WITH MISMATCHED COUNTS:\n")
    for m in mismatches:
        print(f"  {m['match']}: {m['images']} images, {m['labels']} labels (diff: {m['difference']:+d})")
    
    print("\n⚠️  This suggests:")
    if all(m['difference'] > 0 for m in mismatches):
        print("  → More images than labels (some images have no annotations)")
    elif all(m['difference'] < 0 for m in mismatches):
        print("  → More labels than images (some labels have no images)")
    else:
        print("  → Mixed - different issues per match")
else:
    print("\n✓ All matches have equal numbers of images and labels")
    print("  (Though they may still be misaligned due to off-by-one error!)")

print("=" * 80)

In [ ]:
"""
Check if number of images matches number of labels for each match
Including all 3 parts of RBK-BODO separately.
"""

print("=" * 80)
print("IMAGE vs LABEL COUNT VERIFICATION")
print("=" * 80)
print("\nChecking if each match has the same number of images and labels...\n")

mismatches = []

def check_match_counts(match_path, match_name):
    """Check image vs label counts for a single match/part."""
    # Find image and label directories
    image_dirs = list(match_path.rglob('images/train'))
    label_dirs = list(match_path.rglob('labels/train'))
    
    if not image_dirs:
        print(f"  ⚠️  No images/train directory found")
        return None
    
    if not label_dirs:
        print(f"  ⚠️  No labels/train directory found")
        return None
    
    # Count files
    image_files = list(image_dirs[0].glob('*.png')) + list(image_dirs[0].glob('*.jpg'))
    label_files = list(label_dirs[0].glob('*.txt'))
    
    num_images = len(image_files)
    num_labels = len(label_files)
    
    print(f"  Images: {num_images:,}")
    print(f"  Labels: {num_labels:,}")
    
    if num_images == num_labels:
        print(f"  ✓ MATCH - Counts are equal")
    else:
        diff = num_images - num_labels
        print(f"  ✗ MISMATCH - Difference: {diff:+d}")
        if diff > 0:
            print(f"     → {abs(diff)} more images than labels")
        else:
            print(f"     → {abs(diff)} more labels than images")
        
        return {
            'match': match_name,
            'images': num_images,
            'labels': num_labels,
            'difference': diff
        }
    
    # Check frame number ranges
    if image_files and label_files:
        image_nums = []
        for img in image_files:
            num = extract_frame_number(img.name)
            if num is not None:
                image_nums.append(num)
        
        label_nums = []
        for lbl in label_files:
            num = extract_frame_number(lbl.name)
            if num is not None:
                label_nums.append(num)
        
        if image_nums and label_nums:
            img_range = (min(image_nums), max(image_nums))
            lbl_range = (min(label_nums), max(label_nums))
            
            print(f"  Image frame range: {img_range[0]:,} to {img_range[1]:,} (span: {img_range[1] - img_range[0] + 1})")
            print(f"  Label frame range: {lbl_range[0]:,} to {lbl_range[1]:,} (span: {lbl_range[1] - lbl_range[0] + 1})")
            
            if img_range != lbl_range:
                print(f"  ⚠️  WARNING: Frame ranges don't match!")
                if img_range[0] != lbl_range[0]:
                    print(f"     Start mismatch: images start at {img_range[0]}, labels start at {lbl_range[0]}")
                if img_range[1] != lbl_range[1]:
                    print(f"     End mismatch: images end at {img_range[1]}, labels end at {lbl_range[1]}")
    
    return None

for match in sorted(matches):
    if 'BODO' in match.name:
        # Handle RBK-BODO specially - check each part
        print(f"📂 {match.name} (multi-part)")
        
        for part_num in [1, 2, 3]:
            part_dir = match / f"part{part_num}"
            if not part_dir.exists():
                print(f"  Part {part_num}: NOT FOUND")
                continue
            
            # Find the actual data directory within the part
            part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
            if not part_data_dirs:
                print(f"  Part {part_num}: No RBK_BODO_PART* directory found")
                continue
            
            print(f"\n  Part {part_num}: {part_data_dirs[0].name}")
            mismatch = check_match_counts(part_data_dirs[0], f"{match.name}/part{part_num}")
            if mismatch:
                mismatches.append(mismatch)
        
        print()
    else:
        # Regular match
        print(f"📂 {match.name}")
        mismatch = check_match_counts(match, match.name)
        if mismatch:
            mismatches.append(mismatch)
        print()

print("=" * 80)
print("SUMMARY")
print("=" * 80)

if mismatches:
    print(f"\n⚠️  FOUND {len(mismatches)} MATCHES/PARTS WITH MISMATCHED COUNTS:\n")
    for m in mismatches:
        print(f"  {m['match']:<30} Images: {m['images']:>5,}  Labels: {m['labels']:>5,}  Diff: {m['difference']:>+4d}")
    
    print(f"\n⚠️  Analysis:")
    more_images = [m for m in mismatches if m['difference'] > 0]
    more_labels = [m for m in mismatches if m['difference'] < 0]
    
    if more_images:
        print(f"  → {len(more_images)} match(es) have MORE images than labels")
        for m in more_images:
            print(f"     {m['match']}: {m['difference']} extra images")
    
    if more_labels:
        print(f"  → {len(more_labels)} match(es) have MORE labels than images")
        for m in more_labels:
            print(f"     {m['match']}: {abs(m['difference'])} extra labels")
else:
    print("\n✓ All matches have equal numbers of images and labels")
    print("  (Though they are MISALIGNED due to the confirmed off-by-one error!)")

print("\n" + "=" * 80)
print("CONFIRMED ISSUES:")
print("=" * 80)
print("1. ✗ OFF-BY-ONE ERROR: frame_N.png should use frame_(N-1).txt")
print("2. ? COUNT MISMATCH: See above for details")
print("\n→ DATASET REGENERATION REQUIRED WITH CORRECT ALIGNMENT")
print("=" * 80)

In [ ]:
"""
Visualize RBK-BODO edge cases - first and last frames
Since each part has 1 more image than labels, let's see what's happening at the boundaries.
"""

import matplotlib.patches as patches

def parse_yolo_label(label_path):
    """Parse YOLO format label file."""
    annotations = []
    if not label_path.exists():
        return annotations
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls, x, y, w, h = map(float, parts[:5])
                annotations.append({
                    'class': int(cls),
                    'x_center': x,
                    'y_center': y,
                    'width': w,
                    'height': h
                })
    return annotations

def draw_image_with_labels_small(ax, image_path, label_path, title):
    """Draw an image with YOLO labels overlaid on a given axis."""
    # Load image
    img = Image.open(image_path)
    img_width, img_height = img.size
    
    # Parse labels
    annotations = parse_yolo_label(label_path)
    
    # Display image
    ax.imshow(img)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
    
    # Class colors and names
    class_colors = {0: 'red', 1: 'cyan', 2: 'yellow'}
    class_names = {0: 'player', 1: 'ball', 2: 'event'}
    
    # Draw bounding boxes
    for ann in annotations:
        x_center = ann['x_center'] * img_width
        y_center = ann['y_center'] * img_height
        box_width = ann['width'] * img_width
        box_height = ann['height'] * img_height
        
        x1 = x_center - box_width / 2
        y1 = y_center - box_height / 2
        
        color = class_colors.get(ann['class'], 'white')
        name = class_names.get(ann['class'], 'unknown')
        
        rect = patches.Rectangle(
            (x1, y1), box_width, box_height,
            linewidth=1.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        ax.text(
            x1, y1 - 5, f"{name}",
            color=color, fontsize=6, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7)
        )
    
    # Add count
    info_text = f"{len(annotations)} objects" if annotations else "NO LABELS!"
    ax.text(
        10, 30, info_text,
        color='white', fontsize=8, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='black', alpha=0.8),
        verticalalignment='top'
    )

def visualize_bodo_edge_cases():
    """
    Visualize first and last frames of each RBK-BODO part.
    Shows frame with both label options to see the mismatch.
    """
    bodo_path = DATASET_ROOT / 'RBK-BODO'
    
    if not bodo_path.exists():
        print("RBK-BODO not found!")
        return
    
    print("=" * 80)
    print("RBK-BODO EDGE CASE VISUALIZATION")
    print("=" * 80)
    print("\nEach part has 1 more image than labels.")
    print("Checking FIRST and LAST frames to understand the pattern.\n")
    
    for part_num in [1, 2, 3]:
        part_dir = bodo_path / f"part{part_num}"
        if not part_dir.exists():
            continue
        
        # Find the data directory
        part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
        if not part_data_dirs:
            continue
        
        data_dir = part_data_dirs[0]
        
        # Find image and label directories
        image_dir = data_dir / 'data' / 'images' / 'train'
        label_dir = data_dir / 'labels' / 'train'
        
        if not image_dir.exists() or not label_dir.exists():
            continue
        
        # Get all frames
        frames = sorted(image_dir.glob('*.png'))
        if not frames:
            continue
        
        print(f"\n{'=' * 80}")
        print(f"PART {part_num}: {data_dir.name}")
        print('=' * 80)
        print(f"Total images: {len(frames)}")
        
        # Get frame numbers
        frame_nums = []
        for f in frames:
            num = extract_frame_number(f.name)
            if num is not None:
                frame_nums.append((num, f))
        
        frame_nums.sort()
        
        if not frame_nums:
            continue
        
        first_num, first_frame = frame_nums[0]
        last_num, last_frame = frame_nums[-1]
        
        print(f"Frame range: {first_num} to {last_num}")
        
        # Create 2x3 grid:
        # Row 1: First frame with label_0, label_1, label missing
        # Row 2: Last frame with label_(last-1), label_last, check if exists
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # === FIRST FRAME ===
        print(f"\nFirst frame: {first_frame.name} (frame {first_num})")
        
        # Option 1: label with same number
        label_same = label_dir / f"{first_frame.stem}.txt"
        print(f"  Label {first_num}: {label_same.name} - {'EXISTS' if label_same.exists() else 'MISSING'}")
        
        # Option 2: label with number-1
        if first_num > 0:
            label_minus1_name = first_frame.name.replace(f"frame_{first_num:06d}", f"frame_{first_num-1:06d}").replace('.png', '.txt')
            label_minus1 = label_dir / label_minus1_name
            print(f"  Label {first_num-1}: {label_minus1.name} - {'EXISTS' if label_minus1.exists() else 'MISSING'}")
        else:
            label_minus1 = None
        
        # Option 3: label with number+1
        label_plus1_name = first_frame.name.replace(f"frame_{first_num:06d}", f"frame_{first_num+1:06d}").replace('.png', '.txt')
        label_plus1 = label_dir / label_plus1_name
        print(f"  Label {first_num+1}: {label_plus1.name} - {'EXISTS' if label_plus1.exists() else 'MISSING'}")
        
        # Draw first frame with different labels
        if label_minus1 and label_minus1.exists():
            draw_image_with_labels_small(axes[0, 0], first_frame, label_minus1, 
                                         f"First frame + label_{first_num-1}\n{first_frame.name} + {label_minus1.name}")
        else:
            axes[0, 0].text(0.5, 0.5, f"Label {first_num-1}\nNOT FOUND", ha='center', va='center', fontsize=12)
            axes[0, 0].set_title(f"First frame + label_{first_num-1}", fontsize=9, fontweight='bold')
            axes[0, 0].axis('off')
        
        if label_same.exists():
            draw_image_with_labels_small(axes[0, 1], first_frame, label_same,
                                         f"First frame + label_{first_num}\n{first_frame.name} + {label_same.name}")
        else:
            axes[0, 1].text(0.5, 0.5, f"Label {first_num}\nNOT FOUND", ha='center', va='center', fontsize=12)
            axes[0, 1].set_title(f"First frame + label_{first_num}", fontsize=9, fontweight='bold')
            axes[0, 1].axis('off')
        
        if label_plus1.exists():
            draw_image_with_labels_small(axes[0, 2], first_frame, label_plus1,
                                         f"First frame + label_{first_num+1}\n{first_frame.name} + {label_plus1.name}")
        else:
            axes[0, 2].text(0.5, 0.5, f"Label {first_num+1}\nNOT FOUND", ha='center', va='center', fontsize=12)
            axes[0, 2].set_title(f"First frame + label_{first_num+1}", fontsize=9, fontweight='bold')
            axes[0, 2].axis('off')
        
        # === LAST FRAME ===
        print(f"\nLast frame: {last_frame.name} (frame {last_num})")
        
        # Option 1: label with number-1
        label_minus1_name = last_frame.name.replace(f"frame_{last_num:06d}", f"frame_{last_num-1:06d}").replace('.png', '.txt')
        label_minus1 = label_dir / label_minus1_name
        print(f"  Label {last_num-1}: {label_minus1.name} - {'EXISTS' if label_minus1.exists() else 'MISSING'}")
        
        # Option 2: label with same number
        label_same = label_dir / f"{last_frame.stem}.txt"
        print(f"  Label {last_num}: {label_same.name} - {'EXISTS' if label_same.exists() else 'MISSING'}")
        
        # Option 3: label with number+1
        label_plus1_name = last_frame.name.replace(f"frame_{last_num:06d}", f"frame_{last_num+1:06d}").replace('.png', '.txt')
        label_plus1 = label_dir / label_plus1_name
        print(f"  Label {last_num+1}: {label_plus1.name} - {'EXISTS' if label_plus1.exists() else 'MISSING'}")
        
        # Draw last frame with different labels
        if label_minus1.exists():
            draw_image_with_labels_small(axes[1, 0], last_frame, label_minus1,
                                         f"Last frame + label_{last_num-1}\n{last_frame.name} + {label_minus1.name}")
        else:
            axes[1, 0].text(0.5, 0.5, f"Label {last_num-1}\nNOT FOUND", ha='center', va='center', fontsize=12)
            axes[1, 0].set_title(f"Last frame + label_{last_num-1}", fontsize=9, fontweight='bold')
            axes[1, 0].axis('off')
        
        if label_same.exists():
            draw_image_with_labels_small(axes[1, 1], last_frame, label_same,
                                         f"Last frame + label_{last_num}\n{last_frame.name} + {label_same.name}")
        else:
            axes[1, 1].text(0.5, 0.5, f"Label {last_num}\nNOT FOUND", ha='center', va='center', fontsize=12, color='red')
            axes[1, 1].set_title(f"Last frame + label_{last_num}\n⚠️ EXPECTED MISSING", fontsize=9, fontweight='bold')
            axes[1, 1].axis('off')
        
        if label_plus1.exists():
            draw_image_with_labels_small(axes[1, 2], last_frame, label_plus1,
                                         f"Last frame + label_{last_num+1}\n{last_frame.name} + {label_plus1.name}")
        else:
            axes[1, 2].text(0.5, 0.5, f"Label {last_num+1}\nNOT FOUND", ha='center', va='center', fontsize=12)
            axes[1, 2].set_title(f"Last frame + label_{last_num+1}", fontsize=9, fontweight='bold')
            axes[1, 2].axis('off')
        
        fig.suptitle(f"RBK-BODO Part {part_num} - First & Last Frames with All Label Options", 
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

visualize_bodo_edge_cases()

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)
print("""
Look at the visualizations above for each RBK-BODO part:

FIRST FRAME (frame_1):
  - If label_0 is correct → Dataset uses label N-1 (confirmed off-by-one)
  - If label_1 is correct → Dataset uses label N (no off-by-one)
  - If label_2 is correct → Something weird is happening!

LAST FRAME (last frame in part):
  - Label with matching number SHOULD BE MISSING (1 extra image)
  - Label with N-1 should exist and be correct
  - This is the frame without a corresponding label

Expected pattern:
  - frame_1 should use label_0 ✓
  - frame_2 should use label_1 ✓
  - ...
  - frame_N should use label_(N-1) ✓
  - Last frame has no matching label (the +1 extra image)
""")

In [ ]:
"""
Debug: Check what's actually happening with RBK-BODO last frames
Let's list the actual files and see the pattern
"""

print("=" * 80)
print("RBK-BODO DETAILED FILE INSPECTION")
print("=" * 80)

bodo_path = DATASET_ROOT / 'RBK-BODO'

for part_num in [1, 2, 3]:
    part_dir = bodo_path / f"part{part_num}"
    if not part_dir.exists():
        continue
    
    part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
    if not part_data_dirs:
        continue
    
    data_dir = part_data_dirs[0]
    image_dir = data_dir / 'data' / 'images' / 'train'
    label_dir = data_dir / 'labels' / 'train'
    
    if not image_dir.exists() or not label_dir.exists():
        continue
    
    print(f"\n{'=' * 80}")
    print(f"PART {part_num}")
    print('=' * 80)
    
    # Get all images
    images = sorted(image_dir.glob('*.png'))
    labels = sorted(label_dir.glob('*.txt'))
    
    print(f"\nTotal images: {len(images)}")
    print(f"Total labels: {len(labels)}")
    
    # Show first 5 and last 5 of each
    print(f"\nFirst 5 images:")
    for img in images[:5]:
        print(f"  {img.name}")
    
    print(f"\nLast 5 images:")
    for img in images[-5:]:
        print(f"  {img.name}")
    
    print(f"\nFirst 5 labels:")
    for lbl in labels[:5]:
        print(f"  {lbl.name}")
    
    print(f"\nLast 5 labels:")
    for lbl in labels[-5:]:
        print(f"  {lbl.name}")
    
    # Extract frame numbers
    image_nums = []
    for img in images:
        num = extract_frame_number(img.name)
        if num is not None:
            image_nums.append(num)
    
    label_nums = []
    for lbl in labels:
        num = extract_frame_number(lbl.name)
        if num is not None:
            label_nums.append(num)
    
    if image_nums:
        print(f"\nImage frame numbers:")
        print(f"  Min: {min(image_nums)}, Max: {max(image_nums)}")
        print(f"  First 5: {sorted(image_nums)[:5]}")
        print(f"  Last 5: {sorted(image_nums)[-5:]}")
    
    if label_nums:
        print(f"\nLabel frame numbers:")
        print(f"  Min: {min(label_nums)}, Max: {max(label_nums)}")
        print(f"  First 5: {sorted(label_nums)[:5]}")
        print(f"  Last 5: {sorted(label_nums)[-5:]}")
    
    # Check if last image has any corresponding label
    if images:
        last_img = images[-1]
        last_img_num = extract_frame_number(last_img.name)
        
        print(f"\n🔍 LAST IMAGE ANALYSIS:")
        print(f"  Image: {last_img.name}")
        print(f"  Frame number: {last_img_num}")
        
        # Check various label possibilities
        for offset in [-2, -1, 0, 1, 2]:
            test_num = last_img_num + offset
            # Construct label name based on image name pattern
            test_label_name = last_img.name.replace(f"frame_{last_img_num:06d}", f"frame_{test_num:06d}").replace('.png', '.txt')
            test_label_path = label_dir / test_label_name
            
            exists = test_label_path.exists()
            print(f"  Label {test_num} ({test_label_name}): {'✓ EXISTS' if exists else '✗ MISSING'}")

print("\n" + "=" * 80)

In [ ]:
"""
COMPREHENSIVE CHECK: Verify every single image has a corresponding label
Check all frames, not just first/last, to find any missing labels in the middle.
"""

print("=" * 80)
print("COMPREHENSIVE LABEL VERIFICATION - ALL FRAMES")
print("=" * 80)
print("\nChecking EVERY image to ensure it has a corresponding label...")
print("Using the confirmed pattern: image_N.png should have label_(N-1).txt\n")

def check_all_labels(match_path, match_name):
    """Check if every image has a corresponding label with N-1 offset."""
    # Find image and label directories
    image_dirs = list(match_path.rglob('images/train'))
    label_dirs = list(match_path.rglob('labels/train'))
    
    if not image_dirs or not label_dirs:
        return None
    
    image_dir = image_dirs[0]
    label_dir = label_dirs[0]
    
    # Get all images
    images = sorted(image_dir.glob('*.png'))
    
    if not images:
        return None
    
    results = {
        'match': match_name,
        'total_images': len(images),
        'images_with_label': 0,
        'images_without_label': 0,
        'missing_label_frames': [],
        'has_gaps': False
    }
    
    # Check each image
    for img_path in images:
        img_num = extract_frame_number(img_path.name)
        if img_num is None:
            continue
        
        # With off-by-one pattern, image N should use label N-1
        expected_label_num = img_num - 1
        expected_label_name = img_path.name.replace(
            f"frame_{img_num:06d}", f"frame_{expected_label_num:06d}"
        ).replace('.png', '.txt')
        expected_label_path = label_dir / expected_label_name
        
        if expected_label_path.exists():
            results['images_with_label'] += 1
        else:
            results['images_without_label'] += 1
            results['missing_label_frames'].append({
                'image': img_path.name,
                'frame_num': img_num,
                'expected_label': expected_label_name
            })
    
    # Check if missing labels are only at the end or scattered throughout
    if results['missing_label_frames']:
        missing_nums = [f['frame_num'] for f in results['missing_label_frames']]
        # If all missing frames are consecutive at the end, it's expected
        # Otherwise, there are gaps in the middle
        max_img_num = max([extract_frame_number(img.name) for img in images if extract_frame_number(img.name) is not None])
        
        # Check if all missing frames are at the end
        expected_end_missing = list(range(max(missing_nums), max_img_num + 1)) if missing_nums else []
        
        if sorted(missing_nums) != expected_end_missing:
            results['has_gaps'] = True
    
    return results

# Check all matches
all_results = []

for match in sorted(matches):
    if 'BODO' in match.name:
        # Check each part separately
        for part_num in [1, 2, 3]:
            part_dir = match / f"part{part_num}"
            if not part_dir.exists():
                continue
            
            part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
            if not part_data_dirs:
                continue
            
            result = check_all_labels(part_data_dirs[0], f"{match.name}/part{part_num}")
            if result:
                all_results.append(result)
    else:
        result = check_all_labels(match, match.name)
        if result:
            all_results.append(result)

# Display results
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

for result in all_results:
    print(f"\n📂 {result['match']}")
    print(f"  Total images: {result['total_images']:,}")
    print(f"  Images WITH label: {result['images_with_label']:,}")
    print(f"  Images WITHOUT label: {result['images_without_label']:,}")
    
    if result['images_without_label'] == 0:
        print(f"  ✓ ALL IMAGES HAVE LABELS!")
    else:
        if result['has_gaps']:
            print(f"  ✗ MISSING LABELS IN THE MIDDLE (GAPS)!")
            print(f"  Missing frames:")
            for missing in result['missing_label_frames'][:10]:  # Show first 10
                print(f"    - Frame {missing['frame_num']}: {missing['image']} → needs {missing['expected_label']}")
            if len(result['missing_label_frames']) > 10:
                print(f"    ... and {len(result['missing_label_frames']) - 10} more")
        else:
            print(f"  ⚠️  Missing labels only at the END (expected with off-by-one)")
            print(f"  Last {result['images_without_label']} frame(s) have no label:")
            for missing in result['missing_label_frames']:
                print(f"    - Frame {missing['frame_num']}: {missing['image']}")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

total_images = sum(r['total_images'] for r in all_results)
total_with_label = sum(r['images_with_label'] for r in all_results)
total_without_label = sum(r['images_without_label'] for r in all_results)
matches_with_gaps = [r for r in all_results if r['has_gaps']]

print(f"\nTotal images across all matches: {total_images:,}")
print(f"Images with labels: {total_with_label:,} ({total_with_label/total_images*100:.1f}%)")
print(f"Images without labels: {total_without_label:,} ({total_without_label/total_images*100:.1f}%)")

if matches_with_gaps:
    print(f"\n⚠️  CRITICAL: {len(matches_with_gaps)} match(es) have GAPS (missing labels in the middle)!")
    for r in matches_with_gaps:
        print(f"  - {r['match']}: {len(r['missing_label_frames'])} missing labels")
    print("\n  → These frames need investigation - they might be corrupted or mislabeled!")
else:
    print(f"\n✓ No gaps detected - all missing labels are at the end (expected)")
    print("  → Safe to proceed with dataset regeneration")
    print("  → We'll skip frames without labels during regeneration")

print("=" * 80)

In [ ]:
"""
Visualize transitions between RBK-BODO parts - IMAGES ONLY
Show the last 2 images of Part N and first image of Part N+1
NO LABELS - just raw images to check visual continuity
"""

import numpy as np
from PIL import Image

def images_are_identical(img_path1, img_path2):
    """Check if two images are pixel-identical."""
    try:
        img1 = Image.open(img_path1)
        img2 = Image.open(img_path2)
        arr1 = np.array(img1)
        arr2 = np.array(img2)
        return np.array_equal(arr1, arr2)
    except:
        return False

print("=" * 80)
print("RBK-BODO PART TRANSITIONS - RAW IMAGES ONLY")
print("=" * 80)
print("\nShowing last 2 frames of each part + first frame of next part")
print("NO LABELS - just checking visual continuity\n")

bodo_path = DATASET_ROOT / 'RBK-BODO'

# Get all images for each part
parts_images = {}
for part_num in [1, 2, 3]:
    part_dir = bodo_path / f"part{part_num}"
    if not part_dir.exists():
        continue
    
    part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
    if not part_data_dirs:
        continue
    
    data_dir = part_data_dirs[0]
    image_dir = data_dir / 'data' / 'images' / 'train'
    
    if image_dir.exists():
        all_images = sorted(image_dir.glob('*.png'))
        parts_images[part_num] = all_images
        print(f"Part {part_num}: {len(all_images)} images found")

# Transition 1: Part 1 → Part 2
print(f"\n{'=' * 80}")
print("TRANSITION: Part 1 → Part 2")
print('=' * 80)

if 1 in parts_images and 2 in parts_images:
    part1_imgs = parts_images[1]
    part2_imgs = parts_images[2]
    
    # Get the 3 images
    img1 = part1_imgs[-2]  # 2nd to last of part 1
    img2 = part1_imgs[-1]  # Last of part 1
    img3 = part2_imgs[0]   # First of part 2
    
    print(f"Image 1: {img1.name} (Part 1, 2nd to last)")
    print(f"Image 2: {img2.name} (Part 1, LAST)")
    print(f"Image 3: {img3.name} (Part 2, FIRST)")
    
    # Check for duplicates
    print(f"\nDuplicate check:")
    print(f"  Image 1 vs 2: {'IDENTICAL ⚠️' if images_are_identical(img1, img2) else 'Different ✓'}")
    print(f"  Image 2 vs 3: {'IDENTICAL ⚠️' if images_are_identical(img2, img3) else 'Different ✓'}")
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    for idx, img_path in enumerate([img1, img2, img3]):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name, fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    
    fig.suptitle('Part 1 → Part 2 Transition', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Transition 2: Part 2 → Part 3
print(f"\n{'=' * 80}")
print("TRANSITION: Part 2 → Part 3")
print('=' * 80)

if 2 in parts_images and 3 in parts_images:
    part2_imgs = parts_images[2]
    part3_imgs = parts_images[3]
    
    # Get the 3 images
    img1 = part2_imgs[-2]  # 2nd to last of part 2
    img2 = part2_imgs[-1]  # Last of part 2
    img3 = part3_imgs[0]   # First of part 3
    
    print(f"Image 1: {img1.name} (Part 2, 2nd to last)")
    print(f"Image 2: {img2.name} (Part 2, LAST)")
    print(f"Image 3: {img3.name} (Part 3, FIRST)")
    
    # Check for duplicates
    print(f"\nDuplicate check:")
    print(f"  Image 1 vs 2: {'IDENTICAL ⚠️' if images_are_identical(img1, img2) else 'Different ✓'}")
    print(f"  Image 2 vs 3: {'IDENTICAL ⚠️' if images_are_identical(img2, img3) else 'Different ✓'}")
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    for idx, img_path in enumerate([img1, img2, img3]):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name, fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    
    fig.suptitle('Part 2 → Part 3 Transition', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 80)
print("WHAT TO LOOK FOR:")
print("=" * 80)
print("""
Visual continuity:
  ✓ If the 3 images flow smoothly (players moving naturally)
    → Parts are from one continuous video
  
  ✗ If there's an abrupt scene change
    → There's a gap/cut between parts

Duplicates:
  ⚠️ If "Image 2 vs 3" shows IDENTICAL
    → Last frame of Part N is same as first of Part N+1
    → We have overlap - need to skip one of these frames
""")
print("=" * 80)

In [ ]:
"""
Visualize transitions between RBK-BODO parts
Check if parts connect properly by showing:
- Last 2 frames of part N + First frame of part N+1
"""

import matplotlib.patches as patches

def parse_yolo_label(label_path):
    """Parse YOLO format label file."""
    annotations = []
    if not label_path.exists():
        return annotations
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls, x, y, w, h = map(float, parts[:5])
                annotations.append({
                    'class': int(cls),
                    'x_center': x,
                    'y_center': y,
                    'width': w,
                    'height': h
                })
    return annotations

def draw_frame_with_label(ax, image_path, label_path, title):
    """Draw an image with YOLO labels overlaid."""
    # Load image
    img = Image.open(image_path)
    img_width, img_height = img.size
    
    # Parse labels
    annotations = parse_yolo_label(label_path)
    
    # Display image
    ax.imshow(img)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=3)
    ax.axis('off')
    
    # Class colors
    class_colors = {0: 'red', 1: 'cyan', 2: 'yellow'}
    
    # Draw bounding boxes
    for ann in annotations:
        x_center = ann['x_center'] * img_width
        y_center = ann['y_center'] * img_height
        box_width = ann['width'] * img_width
        box_height = ann['height'] * img_height
        
        x1 = x_center - box_width / 2
        y1 = y_center - box_height / 2
        
        color = class_colors.get(ann['class'], 'white')
        
        rect = patches.Rectangle(
            (x1, y1), box_width, box_height,
            linewidth=1.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
    
    # Add count
    info_text = f"{len(annotations)} obj" if annotations else "NO LABELS!"
    ax.text(
        10, 25, info_text,
        color='white', fontsize=8, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.8),
        verticalalignment='top'
    )

print("=" * 80)
print("RBK-BODO PART TRANSITIONS")
print("=" * 80)
print("\nVisualize continuity between parts:")
print("  - Last 2 frames of Part N")
print("  - First frame of Part N+1")
print("\nThis will show if parts connect smoothly or have gaps/discontinuities.\n")

bodo_path = DATASET_ROOT / 'RBK-BODO'

# Get data for all 3 parts
parts_data = {}
for part_num in [1, 2, 3]:
    part_dir = bodo_path / f"part{part_num}"
    if not part_dir.exists():
        continue
    
    part_data_dirs = list(part_dir.glob('RBK_BODO_PART*'))
    if not part_data_dirs:
        continue
    
    data_dir = part_data_dirs[0]
    image_dir = data_dir / 'data' / 'images' / 'train'
    label_dir = data_dir / 'labels' / 'train'
    
    if image_dir.exists() and label_dir.exists():
        images = sorted(image_dir.glob('*.png'))
        parts_data[part_num] = {
            'image_dir': image_dir,
            'label_dir': label_dir,
            'images': images
        }

# Visualize transitions
transitions = [
    (1, 2, "Part 1 → Part 2"),
    (2, 3, "Part 2 → Part 3")
]

for part_a, part_b, title in transitions:
    if part_a not in parts_data or part_b not in parts_data:
        print(f"\n⚠️  Cannot visualize {title} - parts not found")
        continue
    
    print(f"\n{'=' * 80}")
    print(f"{title}")
    print('=' * 80)
    
    # Get frames
    part_a_images = parts_data[part_a]['images']
    part_b_images = parts_data[part_b]['images']
    
    if len(part_a_images) < 2 or len(part_b_images) < 1:
        print(f"Not enough images in parts")
        continue
    
    # Last 2 of part A, first 1 of part B
    frames_to_show = [
        (part_a, part_a_images[-2], f"Part {part_a} - 2nd to last"),
        (part_a, part_a_images[-1], f"Part {part_a} - LAST"),
        (part_b, part_b_images[0], f"Part {part_b} - FIRST"),
    ]
    
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for idx, (part_num, img_path, desc) in enumerate(frames_to_show):
        frame_num = extract_frame_number(img_path.name)
        label_dir = parts_data[part_num]['label_dir']
        
        # Expected label with off-by-one pattern
        expected_label_num = frame_num - 1
        expected_label_name = img_path.name.replace(
            f"frame_{frame_num:06d}", f"frame_{expected_label_num:06d}"
        ).replace('.png', '.txt')
        expected_label_path = label_dir / expected_label_name
        
        print(f"\n{desc}:")
        print(f"  Image: {img_path.name} (frame {frame_num})")
        print(f"  Expected label: {expected_label_name} - {'✓ EXISTS' if expected_label_path.exists() else '✗ MISSING'}")
        
        if expected_label_path.exists():
            draw_frame_with_label(
                axes[idx], img_path, expected_label_path,
                f"{desc}\n{img_path.name} + {expected_label_name}"
            )
        else:
            axes[idx].text(0.5, 0.5, 
                          f"{desc}\n{img_path.name}\nLabel MISSING!\n{expected_label_name}", 
                          ha='center', va='center', fontsize=10, 
                          color='red', fontweight='bold')
            axes[idx].set_title(f"{desc}\n{img_path.name} (NO LABEL!)", 
                               fontsize=9, fontweight='bold')
            axes[idx].axis('off')
    
    fig.suptitle(f"RBK-BODO Transition: {title}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)
print("""
Look at the visualizations above:

✓ If frames look continuous (similar scene, player positions):
  → Parts were split from a continuous video
  → No temporal gap between parts

✗ If frames look discontinuous (completely different scenes):
  → Parts are from different segments of the match
  → There's a temporal gap between parts

⚠️  If last frames of Part N are missing labels:
  → Confirms the count mismatch we found earlier
  → Those frames should be skipped in dataset regeneration
  
This helps us understand if we can treat all 3 parts as one continuous
sequence or if they're separate segments.
""")
print("=" * 80)

In [ ]:
"""
Check for gaps in label sequences for RBK-BODO part 3
List all labels and find any missing numbers
"""

print("=" * 80)
print("RBK-BODO PART 3 - LABEL SEQUENCE GAP CHECK")
print("=" * 80)

bodo_path = DATASET_ROOT / 'RBK-BODO'
part3_dir = bodo_path / 'part3'

if not part3_dir.exists():
    print("Part 3 not found!")
else:
    # Find data directory
    part_data_dirs = list(part3_dir.glob('RBK_BODO_PART*'))
    if not part_data_dirs:
        print("No RBK_BODO_PART* directory found!")
    else:
        data_dir = part_data_dirs[0]
        label_dir = data_dir / 'labels' / 'train'
        
        if not label_dir.exists():
            print(f"Labels directory not found: {label_dir}")
        else:
            # Get all label files
            labels = sorted(label_dir.glob('*.txt'))
            
            print(f"\nLabel directory: {label_dir}")
            print(f"Total label files: {len(labels)}")
            
            # Extract frame numbers
            label_nums = []
            for lbl in labels:
                num = extract_frame_number(lbl.name)
                if num is not None:
                    label_nums.append(num)
            
            label_nums.sort()
            
            if not label_nums:
                print("Could not extract frame numbers!")
            else:
                min_num = min(label_nums)
                max_num = max(label_nums)
                
                print(f"\nLabel frame range: {min_num} to {max_num}")
                print(f"Expected count: {max_num - min_num + 1}")
                print(f"Actual count: {len(label_nums)}")
                
                # Find missing numbers
                expected_set = set(range(min_num, max_num + 1))
                actual_set = set(label_nums)
                missing = sorted(expected_set - actual_set)
                
                if missing:
                    print(f"\n⚠️  MISSING LABELS: {len(missing)} gaps found!")
                    print(f"\nMissing frame numbers:")
                    
                    # Group consecutive missing numbers
                    if len(missing) <= 50:
                        # Show all if not too many
                        for num in missing:
                            print(f"  - frame_{num:06d}.txt")
                    else:
                        # Show first 20 and last 20
                        print("  First 20 missing:")
                        for num in missing[:20]:
                            print(f"    - frame_{num:06d}.txt")
                        print(f"  ... {len(missing) - 40} more ...")
                        print("  Last 20 missing:")
                        for num in missing[-20:]:
                            print(f"    - frame_{num:06d}.txt")
                    
                    # Check if gaps are clustered or scattered
                    gaps = []
                    for i in range(len(missing) - 1):
                        if missing[i+1] - missing[i] > 1:
                            gaps.append((missing[i], missing[i+1]))
                    
                    if gaps:
                        print(f"\n  Gaps are SCATTERED (found {len(gaps)} separate clusters)")
                    else:
                        print(f"\n  All missing labels are CONSECUTIVE")
                else:
                    print(f"\n✓ NO GAPS - All numbers from {min_num} to {max_num} are present!")
                
                # Show distribution
                print(f"\n" + "─" * 80)
                print("LABEL SEQUENCE DETAILS")
                print("─" * 80)
                print(f"First 10 labels: {label_nums[:10]}")
                print(f"Last 10 labels: {label_nums[-10:]}")
                
                # Check for duplicates
                duplicates = [num for num in label_nums if label_nums.count(num) > 1]
                if duplicates:
                    print(f"\n⚠️  DUPLICATE frame numbers found: {set(duplicates)}")
                else:
                    print(f"\n✓ No duplicates")

print("=" * 80)